In [ ]:
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore, BDeuScore
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from joblib import Parallel, delayed
from tqdm import tqdm
from tqdm_joblib import tqdm_joblib  # Import tqdm_joblib
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN  # Import oversampling methods

# Suppress warnings
warnings.filterwarnings("ignore", category=FutureWarning)

def discretize_binary_global(X_df):
    """
    Discretize continuous features into binary based on the global median.
    """
    X_discretized = X_df.copy()
    for column in X_discretized.columns:
        median = X_discretized[column].median()
        X_discretized[column] = (X_discretized[column] > median).astype(int)
    return X_discretized

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")  # Ensure file path is correct

# Extract predictors (X) and outcome (Y)
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Apply global discretization
X_discretized = discretize_binary_global(X)

# Combine discretized features with the outcome
data_discretized = X_discretized.copy()
data_discretized['RRI'] = Y

# Select features based on feature selection
selected_feature_indexes = [254, 224, 3, 208, 241, 202, 25, 67, 74, 210, 216, 60, 246, 245, 53, 66, 48, 33, 57, 229, 212, 81, 88, 102, 7, 78, 58, 244, 207, 181, 112, 232, 76, 197, 13, 52]
selected_feature_names = X_discretized.columns[selected_feature_indexes].tolist()

# Extract selected features
X_selected = X_discretized[selected_feature_names]
data_selected = X_selected.copy()
data_selected['RRI'] = Y

# Initialize 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring methods
scoring_methods = {
    'BIC': BicScore(data_selected),
    'BDeu_1': BDeuScore(data_selected, equivalent_sample_size=1),
    'BDeu_2': BDeuScore(data_selected, equivalent_sample_size=2),
    'BDeu_5': BDeuScore(data_selected, equivalent_sample_size=5)
}

# Define hyperparameter grid
max_parents_options = [None, 2, 3]
oversamplers = ['passthrough', 'RandomOverSampler', 'SMOTE', 'ADASYN']
sampling_rates = [0.2, 0.5, 0.7, 1.0]

hyperparameter_grid = []
for scoring_method in scoring_methods.keys():
    for max_parents in max_parents_options:
        hyperparameter_grid.append({
            'scoring_method': scoring_method,
            'max_parents': max_parents,
            'oversampler': 'passthrough',
            'sampling_rate': None
        })
        for oversampler in ['RandomOverSampler', 'SMOTE', 'ADASYN']:
            for sampling_rate in sampling_rates:
                hyperparameter_grid.append({
                    'scoring_method': scoring_method,
                    'max_parents': max_parents,
                    'oversampler': oversampler,
                    'sampling_rate': sampling_rate
                })

print(f"Total hyperparameter combinations: {len(hyperparameter_grid)}")

def evaluate_hyperparameters(params, data, selected_features, kf, scoring_methods):
    scoring_method_name = params['scoring_method']
    max_parents = params['max_parents']
    oversampler_name = params['oversampler']
    sampling_rate = params['sampling_rate']
    scoring_method = scoring_methods[scoring_method_name]
    
    if oversampler_name == 'passthrough':
        oversampler = None
    elif oversampler_name == 'RandomOverSampler':
        oversampler = RandomOverSampler(sampling_strategy=sampling_rate, random_state=42)
    elif oversampler_name == 'SMOTE':
        oversampler = SMOTE(sampling_strategy=sampling_rate, random_state=42)
    elif oversampler_name == 'ADASYN':
        oversampler = ADASYN(sampling_strategy=sampling_rate, random_state=42)
    else:
        raise ValueError(f"Unknown oversampler: {oversampler_name}")
    
    auc_scores = []
    for fold, (train_index, test_index) in enumerate(kf.split(data), 1):
        train_data = data.iloc[train_index].copy()
        test_data = data.iloc[test_index].copy()

        X_train = train_data[selected_features]
        Y_train = train_data['RRI']
        X_test = test_data[selected_features]
        Y_test = test_data['RRI']

        if oversampler is not None:
            try:
                X_train_resampled, Y_train_resampled = oversampler.fit_resample(X_train, Y_train)
                train_data_resampled = X_train_resampled.copy()
                train_data_resampled['RRI'] = Y_train_resampled
            except Exception as e:
                print(f"    Fold {fold}: Oversampling failed with error: {e}")
                continue
        else:
            train_data_resampled = train_data.copy()

        hc = HillClimbSearch(train_data_resampled)
        try:
            if max_parents is not None:
                best_model_structure = hc.estimate(max_indegree=max_parents, scoring_method=scoring_method)
            else:
                best_model_structure = hc.estimate(scoring_method=scoring_method)
        except Exception as e:
            print(f"    Fold {fold}: Structure learning failed with error: {e}")
            continue

        model = BayesianNetwork(best_model_structure.edges())
        try:
            model.fit(train_data_resampled, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=10)
        except Exception as e:
            print(f"    Fold {fold}: Model fitting failed with error: {e}")
            continue

        infer = VariableElimination(model)

        y_true = test_data['RRI']
        y_pred_probs = []
        network_vars = set(model.nodes())
        for _, row in test_data.iterrows():
            evidence = {col: row[col] for col in selected_features if col in network_vars and col != 'RRI'}
            try:
                result = infer.query(variables=['RRI'], evidence=evidence, show_progress=False)
                y_pred_probs.append(result.values[1])
            except Exception:
                y_pred_probs.append(0.0)

        try:
            auc = roc_auc_score(y_true, y_pred_probs)
            auc_scores.append(auc)
        except ValueError:
            auc_scores.append(0.5)

    avg_auc = np.mean(auc_scores) if auc_scores else 0
    std_auc = np.std(auc_scores) if auc_scores else 0

    return {
        'scoring_method': scoring_method_name,
        'max_parents': max_parents,
        'oversampler': oversampler_name,
        'sampling_rate': sampling_rate,
        'avg_auc': avg_auc,
        'std_auc': std_auc
    }

def evaluate_all_hyperparameters(grid, data, selected_features, kf, scoring_methods):
    results = []
    with tqdm_joblib(tqdm(desc="Hyperparameter Tuning", total=len(grid))):
        results = Parallel(n_jobs=-1, verbose=0)(
            delayed(evaluate_hyperparameters)(params, data, selected_features, kf, scoring_methods) for params in grid
        )
    return results

results_hyper = evaluate_all_hyperparameters(hyperparameter_grid, data_selected, selected_feature_names, kf, scoring_methods)
results_hyper_df = pd.DataFrame(results_hyper)
results_hyper_df = results_hyper_df[results_hyper_df['avg_auc'] > 0]

if not results_hyper_df.empty:
    best_hyper = results_hyper_df.loc[results_hyper_df['avg_auc'].idxmax()]
    print("\nBest Hyperparameters Identified:")
    print(best_hyper)
else:
    print("\nNo valid hyperparameter tuning results were obtained.")

/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


Total hyperparameter combinations: 156


Hyperparameter Tuning:   0%|          | 0/156 [00:00<?, ?it/s]

  0%|          | 0/156 [00:00<?, ?it/s]

/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
  0%|          | 0/1000000 [00:00<?, ?it/s]/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux